In [196]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly import graph_objects as go
from sklearn.metrics import classification_report, confusion_matrix
import tiktoken
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchinfo import summary

In [209]:
df_spam = pd.read_csv('../datas/spam_clean.csv', encoding='latin-1')
df_spam.head()

,label_string,message,label
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
tokenizer = tiktoken.get_encoding("cl100k_base")

def encode_texts(texts):
    return [tokenizer.encode(text) for text in texts]

tokens = encode_texts(df_spam["message"])

In [211]:
#Let's check the first line of tokens, from a ham message
tokens[0][:10]

[11087, 3156, 16422, 647, 1486, 11, 14599, 497, 16528, 1193]

In [212]:
#Let's check the third line of tokens, from a spam message
tokens[2][:10]

[11180, 4441, 304, 220, 17, 264, 74860, 398, 1391, 311]

In [213]:
seq_lens = [len(seq) for seq in tokens]
np.mean(seq_lens)

np.float64(22.893933955491743)

In [214]:
fig = px.histogram(seq_lens, nbins=30, color_discrete_sequence=px.colors.qualitative.Antique)
fig.update_layout(title="Distribution of the sequences' lengths", 
                  yaxis_title="Nb of sequences", xaxis_title="Size of the sequences", title_x=0.5, showlegend=False)
fig.show()

In [216]:
mask_spam = df_spam["label_string"] == "spam"
tokens_spam = encode_texts(df_spam.loc[mask_spam, "message"])

mask_ham = df_spam["label_string"] == "ham"
tokens_ham = encode_texts(df_spam.loc[mask_ham, "message"])

print(f'Average length of spam messages: {np.mean([len(seq) for seq in tokens_spam])}')
print()
print(f'Average length of ham messages: {np.mean([len(seq) for seq in tokens_ham])}')

Average length of spam messages: 42.900937081659976

Average length of ham messages: 19.796476683937822


Spam messages tend to be way longer on average than ham messages. But in order to fight a potential overfitting of the model, we won't take it into account at first. We might return to it later for more adjustments.

In [217]:
#As the average length is around 23, we will pad/truncate all sequences to 30 tokens.
def pad_sequences(sequences, max_length=30):
    return [seq[:max_length] + [0] * (max_length - len(seq)) for seq in sequences]

tokens = pad_sequences(tokens)

In [218]:
# Class ATTDataset
class ATTDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

df_dataset = ATTDataset(tokens, df_spam["label"])

# Split dataset into training (80%) and validation (20%)
train_size = int(0.8 * len(df_dataset))
val_size = len(df_dataset) - train_size
train_dataset, val_dataset = random_split(df_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [219]:
text, label = next(iter(train_loader))
print(label)
print(text)

tensor([0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([[ 2822, 70470,  5954,     0,  2650,   577,   656,   258,   266,   279,
          4647,    30,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [   43,   337,  2564,   358,  2216,  1205,   311,  6227,   311,  8343,
           994,   358,  2846, 16558,   719,   358,   656, 15763,   499, 10494,
           757,  2883,   430,  3814, 41346,   353,  3647,  3742,     9,     0],
        [24205,   297,  2641,   579,   514,   497,    84,  2586,   220,    17,
            76,  7924,  4983,    64,   497,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [24220,  1093,   422,   433,  5900,  1093,   433,  1550,   449,   856,
          4885,   737,  1764, 18791,   856, 17619,   304,  109

Let's start making predictions

In [220]:
vocab_size = tokenizer.n_vocab

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(embed_dim, num_class)

    def forward(self, text):
        embedded = self.embedding(text)
        pooled = self.pooling(embedded.permute(0, 2, 1)).squeeze(2)
        return torch.sigmoid(self.fc(pooled))

model = TextClassifier(vocab_size=vocab_size,
                      embed_dim=16,
                      num_class=1)

In [221]:
print(model)

# Print model summary
summary(model, input_data=text)

TextClassifier(
  (embedding): Embedding(100277, 16, padding_idx=0)
  (pooling): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)


Layer (type:depth-idx)                   Output Shape              Param #
TextClassifier                           [32, 1]                   --
├─Embedding: 1-1                         [32, 30, 16]              1,604,432
├─AdaptiveAvgPool1d: 1-2                 [32, 16, 1]               --
├─Linear: 1-3                            [32, 1]                   17
Total params: 1,604,449
Trainable params: 1,604,449
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 51.34
Input size (MB): 0.01
Forward/backward pass size (MB): 0.12
Params size (MB): 6.42
Estimated Total Size (MB): 6.55

In [222]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):

    # Dictionary to store training & validation loss and accuracy over epochs
    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}

    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions

        # Training loop
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # Reset gradients before each batch
            outputs = model(inputs).squeeze()  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation (compute gradients)
            optimizer.step()  # Update model parameters

            total_loss += loss.item()  # Accumulate batch loss
            correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)

        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        with torch.no_grad():  # No need to compute gradients during validation
            for inputs, labels in val_loader:
                outputs = model(inputs).squeeze()  # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss
                val_correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)

        # Store metrics in history dictionary
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return history  # Return training history

history = train(model,
                train_loader=train_loader,
                val_loader=val_loader,
                criterion=criterion,
                optimizer=optimizer,
                epochs=20)

Epoch [1/20], Loss: 0.6547, Acc: 0.7669, Val Loss: 0.6093, Val Acc: 0.8700
Epoch [2/20], Loss: 0.5574, Acc: 0.8685, Val Loss: 0.5028, Val Acc: 0.8744
Epoch [3/20], Loss: 0.4505, Acc: 0.8802, Val Loss: 0.3986, Val Acc: 0.8924
Epoch [4/20], Loss: 0.3560, Acc: 0.9002, Val Loss: 0.3176, Val Acc: 0.9184
Epoch [5/20], Loss: 0.2849, Acc: 0.9248, Val Loss: 0.2576, Val Acc: 0.9363
Epoch [6/20], Loss: 0.2307, Acc: 0.9417, Val Loss: 0.2130, Val Acc: 0.9552
Epoch [7/20], Loss: 0.1885, Acc: 0.9612, Val Loss: 0.1803, Val Acc: 0.9632
Epoch [8/20], Loss: 0.1573, Acc: 0.9717, Val Loss: 0.1558, Val Acc: 0.9713
Epoch [9/20], Loss: 0.1341, Acc: 0.9782, Val Loss: 0.1376, Val Acc: 0.9740
Epoch [10/20], Loss: 0.1156, Acc: 0.9825, Val Loss: 0.1236, Val Acc: 0.9740
Epoch [11/20], Loss: 0.1014, Acc: 0.9861, Val Loss: 0.1126, Val Acc: 0.9776
Epoch [12/20], Loss: 0.0895, Acc: 0.9868, Val Loss: 0.1037, Val Acc: 0.9812
Epoch [13/20], Loss: 0.0813, Acc: 0.9879, Val Loss: 0.0962, Val Acc: 0.9821
Epoch [14/20], Loss: 

In [223]:
checkpoint_path = "../models/AT_T_Spam_detection_model.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "history": history,
}, checkpoint_path)

Results

In [224]:
color_chart = [
    "#D9534F", 
    "#5CB85C",  
    "#C9302C", 
    "#449D44",  
    "#F2B6B3",  
    "#B7E1B0",  
    "#7A1C1C",  
    "#1F5F1F"   
]

fig = go.Figure(data=[
                      go.Scatter(
                          y=history["loss"],
                          name="Training loss",
                          mode="lines",
                          marker=dict(
                              color=color_chart[0]
                          )),
                      go.Scatter(
                          y=history["val_loss"],
                          name="Validation loss",
                          mode="lines",
                          marker=dict(
                              color=color_chart[1]
                          ))
])
fig.update_layout(
    title="Training and val loss across epochs",
    xaxis_title="epochs",
    yaxis_title="Cross Entropy"
)
fig.show()

In [225]:
color_chart = [
    "#D9534F", 
    "#5CB85C",  
    "#C9302C", 
    "#449D44",  
    "#F2B6B3",  
    "#B7E1B0",  
    "#7A1C1C",  
    "#1F5F1F"   
]

fig = go.Figure(data=[
                      go.Scatter(
                          y=history["accuracy"],
                          name="Training Accuracy",
                          mode="lines",
                          marker=dict(
                              color=color_chart[0]
                          )),
                      go.Scatter(
                          y=history["val_accuracy"],
                          name="Validation Accuracy",
                          mode="lines",
                          marker=dict(
                              color=color_chart[1]
                          ))
])
fig.update_layout(
    title="Training and val Accuracy across epochs",
    xaxis_title="epochs",
    yaxis_title="Cross Entropy"
)
fig.show()

In [226]:
result_history = {key: valeur[-1] for key, valeur in history.items()}
print(result_history)

{'loss': 0.04032074430558298, 'val_loss': 0.06817278558654445, 'accuracy': 0.9934933811981154, 'val_accuracy': 0.9865470852017937}


In [227]:
# Function to evaluate the model and get worst predictions
def evaluate_worst_predictions(model, dataloader, tokenizer):
    # Set model to evaluation mode to disable dropout and batch normalization
    model.eval()

    # Lists to store all predictions, labels, errors, and inputs for analysis
    list_predictions = []
    list_labels = []
    list_errors = []
    list_inputs = []

    # No gradients needed during evaluation for efficiency
    with torch.no_grad():
        for batch in dataloader:
            # Extract inputs and labels from the batch
            inputs, labels = batch
            outputs = model(inputs) # Forward pass: Get model predictions

            # Convert outputs to predicted class for classification problems
            #preds = torch.argmax(outputs, dim=1)
            preds = (outputs >= 0.5).int().squeeze()
            errors = (preds != labels).float()  # Misclassified observations
            
            # Store predictions, labels, errors, and raw inputs for further analysis
            list_predictions.extend(preds.cpu().numpy())
            list_labels.extend(int(x) for x in labels.cpu().numpy())
            list_errors.extend(errors.cpu().numpy())
            list_inputs.extend(inputs.cpu().numpy())

    # Convert stored results into a Pandas DataFrame for easy analysis
    # Decode tokenized text back into human-readable text
    df_results = pd.DataFrame({
        "True_Label": list_labels,
        "Predicted": list_predictions,
        "Error": list_errors,
        "Inputs": list_inputs,
        "Text" : [tokenizer.decode(input) for input in list_inputs]
    })

    # Sort the DataFrame by highest error to identify the worst predictions
    df_results_sorted = df_results.sort_values(by="Error", ascending=False)

    # Return the sorted DataFrame containing worst predictions
    return df_results_sorted

# Evaluate worst predictions on validation set
worst_predictions_val = evaluate_worst_predictions(model, val_loader, tokenizer)

# Evaluate worst predictions on training set
worst_predictions_train = evaluate_worst_predictions(model, train_loader, tokenizer)

In [228]:
worst_predictions_train["Predicted"].value_counts()

Predicted
0    3878
1     579
Name: count, dtype: int64

In [229]:
worst_predictions_train.head(10)

,True_Label,Predicted,Error,Inputs,Text
2292,1,0,1.0,"[11180, 6749, 28653, 1070, 76745, 433, 596, 10...",FreeMsg Hey there darling it's been 3 week's n...
1507,0,1,1.0,"[2028, 374, 4433, 3663, 1296, 320, 220, 16, 22...",This is ur face test ( 1 2 3 4 5 6 7 8 9 &lt;#&
2336,1,0,1.0,"[14442, 15007, 617, 1047, 1403, 315, 1521, 13,...",dating:i have had two of these. Only started a...
2767,1,0,1.0,"[33562, 1148, 433, 5097, 220, 17, 1935, 961, 3...",Got what it takes 2 take part in the WRC Rally...
1428,1,0,1.0,"[29089, 499, 1093, 311, 1518, 856, 20572, 2206...",Would you like to see my XXX pics they are so ...
2673,1,0,1.0,"[13347, 13835, 41346, 865, 577, 220, 19, 23196...",Hi ya babe x u 4goten bout me?' scammers getti...
320,1,0,1.0,"[2028, 1984, 374, 7263, 311, 499, 555, 19722, ...",This message is brought to you by GMW Ltd. and...
2979,1,0,1.0,"[7131, 499, 6865, 922, 279, 502, 1144, 12792, ...","Did you hear about the new \Divorce Barbie\""? ..."
3657,1,0,1.0,"[2675, 2834, 956, 4510, 433, 719, 433, 596, 83...",You won't believe it but it's true. It's Incre...
2683,1,0,1.0,"[25821, 602, 617, 2834, 3243, 287, 1396, 220, ...",Money i have won wining number 946 wot do i do...


In [230]:
print(classification_report(worst_predictions_train["True_Label"], worst_predictions_train["Predicted"]))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3853
           1       1.00      0.96      0.98       604

    accuracy                           0.99      4457
   macro avg       1.00      0.98      0.99      4457
weighted avg       0.99      0.99      0.99      4457



In [ ]:
mat = confusion_matrix(worst_predictions_train["True_Label"], worst_predictions_train["Predicted"])

labels = df_spam["label_string"].unique()
df_mat = pd.DataFrame(mat, index=labels, columns=labels)

fig = px.imshow(df_mat, text_auto=True, color_continuous_scale=px.colors.sequential.Agsunset,
    labels=dict(x="Prediction", y="Real", color="Nombre"))
fig.update_coloraxes(showscale=False)
fig.show()

In [232]:
worst_predictions_val.head(10) #Spam = 1 ; Ham = 0

,True_Label,Predicted,Error,Inputs,Text
669,0,1,1.0,"[40, 320, 91293, 23683, 8, 617, 3779, 577, 439...",I (Career Tel) have added u as a contact on IN...
670,1,0,1.0,"[11787, 499, 5016, 3403, 30, 7531, 704, 505, 2...",Are you unique enough? Find out from 30th Augu...
727,1,0,1.0,"[33562, 1148, 433, 5097, 220, 17, 1935, 961, 3...",Got what it takes 2 take part in the WRC Rally...
861,1,0,1.0,"[6509, 1026, 8855, 916, 15714, 499, 3708, 1949...",thesmszone.com lets you send free anonymous an...
124,1,0,1.0,"[4438, 2586, 433, 5097, 779, 2697, 892, 369, 2...",How come it takes so little time for a child w...
854,1,0,1.0,"[13347, 41346, 1202, 17527, 11, 1268, 436, 577...","Hi babe its Jordan, how r u? Im home from abro..."
853,1,0,1.0,"[3648, 350, 3233, 1115, 2046, 2997, 25, 220, 1...","New Tones This week include: 1)McFly-All Ab..,..."
84,1,0,1.0,"[2675, 2834, 956, 4510, 433, 719, 433, 596, 83...",You won't believe it but it's true. It's Incre...
495,0,1,1.0,"[7072, 430, 220, 18, 0, 220, 19, 55463, 25491,...",make that 3! 4 fucks sake?! x!!!!!!!!!!!!!!!!!!!
492,1,0,1.0,"[17400, 279, 24519, 46433, 126, 231, 19321, 12...",Win the newest ÂÃÃHarry Potter and the Orde...


In [233]:
print(classification_report(worst_predictions_val["True_Label"], worst_predictions_val["Predicted"]))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       972
           1       0.98      0.92      0.95       143

    accuracy                           0.99      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [ ]:
mat = confusion_matrix(worst_predictions_val["True_Label"], worst_predictions_val["Predicted"])

labels = df_spam["label_string"].unique()
df_mat = pd.DataFrame(mat, index=labels, columns=labels)

fig = px.imshow(df_mat, text_auto=True, color_continuous_scale=px.colors.sequential.Agsunset,
    labels=dict(x="Prediction", y="Real", color="Nombre"))
fig.update_coloraxes(showscale=False)
fig.show()

Now let's try to adjust the pad sequence to the average length of spam messages. We had 30, but still in order not to overfit, we will just increase it slightly. At 35.

In [235]:
def pad_sequences_plus(sequences, max_length=35):
    return [seq[:max_length] + [0] * (max_length - len(seq)) for seq in sequences]

tokens_plus = pad_sequences_plus(tokens)

In [237]:
df_dataset_adjusted = ATTDataset(tokens_plus, df_spam["label"])

# Split dataset into training (80%) and validation (20%)
train_size = int(0.8 * len(df_dataset))
val_size = len(df_dataset) - train_size
train_dataset, val_dataset = random_split(df_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [238]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):

    # Dictionary to store training & validation loss and accuracy over epochs
    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}

    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions

        # Training loop
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # Reset gradients before each batch
            outputs = model(inputs).squeeze()  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation (compute gradients)
            optimizer.step()  # Update model parameters

            total_loss += loss.item()  # Accumulate batch loss
            correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)

        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        with torch.no_grad():  # No need to compute gradients during validation
            for inputs, labels in val_loader:
                outputs = model(inputs).squeeze()  # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss
                val_correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)

        # Store metrics in history dictionary
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return history  # Return training history

history = train(model,
                train_loader=train_loader,
                val_loader=val_loader,
                criterion=criterion,
                optimizer=optimizer,
                epochs=20)

Epoch [1/20], Loss: 0.0444, Acc: 0.9915, Val Loss: 0.0314, Val Acc: 0.9946
Epoch [2/20], Loss: 0.0400, Acc: 0.9924, Val Loss: 0.0283, Val Acc: 0.9955
Epoch [3/20], Loss: 0.0358, Acc: 0.9935, Val Loss: 0.0259, Val Acc: 0.9955
Epoch [4/20], Loss: 0.0324, Acc: 0.9939, Val Loss: 0.0239, Val Acc: 0.9955
Epoch [5/20], Loss: 0.0295, Acc: 0.9944, Val Loss: 0.0222, Val Acc: 0.9955
Epoch [6/20], Loss: 0.0269, Acc: 0.9946, Val Loss: 0.0208, Val Acc: 0.9955
Epoch [7/20], Loss: 0.0245, Acc: 0.9946, Val Loss: 0.0196, Val Acc: 0.9955
Epoch [8/20], Loss: 0.0223, Acc: 0.9951, Val Loss: 0.0185, Val Acc: 0.9955
Epoch [9/20], Loss: 0.0203, Acc: 0.9951, Val Loss: 0.0176, Val Acc: 0.9955
Epoch [10/20], Loss: 0.0187, Acc: 0.9951, Val Loss: 0.0168, Val Acc: 0.9955
Epoch [11/20], Loss: 0.0169, Acc: 0.9951, Val Loss: 0.0160, Val Acc: 0.9955
Epoch [12/20], Loss: 0.0155, Acc: 0.9966, Val Loss: 0.0154, Val Acc: 0.9955
Epoch [13/20], Loss: 0.0141, Acc: 0.9969, Val Loss: 0.0148, Val Acc: 0.9955
Epoch [14/20], Loss: 

In [239]:
checkpoint_path = "../models/AT_T_Spam_detection_model_v2.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "history": history,
}, checkpoint_path)

In [240]:
color_chart = [
    "#D9534F", 
    "#5CB85C",  
    "#C9302C", 
    "#449D44",  
    "#F2B6B3",  
    "#B7E1B0",  
    "#7A1C1C",  
    "#1F5F1F"   
]

fig = go.Figure(data=[
                      go.Scatter(
                          y=history["loss"],
                          name="Training loss",
                          mode="lines",
                          marker=dict(
                              color=color_chart[0]
                          )),
                      go.Scatter(
                          y=history["val_loss"],
                          name="Validation loss",
                          mode="lines",
                          marker=dict(
                              color=color_chart[1]
                          ))
])
fig.update_layout(
    title="Training and val loss across epochs",
    xaxis_title="epochs",
    yaxis_title="Cross Entropy"
)
fig.show()

In [241]:
color_chart = [
    "#D9534F", 
    "#5CB85C",  
    "#C9302C", 
    "#449D44",  
    "#F2B6B3",  
    "#B7E1B0",  
    "#7A1C1C",  
    "#1F5F1F"   
]

fig = go.Figure(data=[
                      go.Scatter(
                          y=history["accuracy"],
                          name="Training Accuracy",
                          mode="lines",
                          marker=dict(
                              color=color_chart[0]
                          )),
                      go.Scatter(
                          y=history["val_accuracy"],
                          name="Validation Accuracy",
                          mode="lines",
                          marker=dict(
                              color=color_chart[1]
                          ))
])
fig.update_layout(
    title="Training and val Accuracy across epochs",
    xaxis_title="epochs",
    yaxis_title="Cross Entropy"
)
fig.show()

It seems that lengthening the pad sequence increases the accuracy drastically. In the former test, by the 20th epoch, val_accuracy was at around 0.980. Now, it is above 0.993. Worth noting given the evolution of the val and train loss, there might be a really minor and negligible overfitting, since the model stops getting better results after the 14th/15th epoch, but the model still has way better results than the previous one.

In [242]:
result_history_model_2 = {key: valeur[-1] for key, valeur in history.items()}
print(result_history_model_2)
print()
#Let's compare with previous results
print(result_history)

{'loss': 0.007298086542868986, 'val_loss': 0.012213806887822492, 'accuracy': 0.9991025353376711, 'val_accuracy': 0.9982062780269059}

{'loss': 0.04032074430558298, 'val_loss': 0.06817278558654445, 'accuracy': 0.9934933811981154, 'val_accuracy': 0.9865470852017937}


In [243]:
# Function to evaluate the model and get worst predictions
def evaluate_worst_predictions(model, dataloader, tokenizer):
    # Set model to evaluation mode to disable dropout and batch normalization
    model.eval()

    # Lists to store all predictions, labels, errors, and inputs for analysis
    list_predictions = []
    list_labels = []
    list_errors = []
    list_inputs = []

    # No gradients needed during evaluation for efficiency
    with torch.no_grad():
        for batch in dataloader:
            # Extract inputs and labels from the batch
            inputs, labels = batch
            outputs = model(inputs) # Forward pass: Get model predictions

            # Convert outputs to predicted class for classification problems
            #preds = torch.argmax(outputs, dim=1)
            preds = (outputs >= 0.5).int().squeeze()
            errors = (preds != labels).float()  # Misclassified observations
            
            # Store predictions, labels, errors, and raw inputs for further analysis
            list_predictions.extend(preds.cpu().numpy())
            list_labels.extend(int(x) for x in labels.cpu().numpy())
            list_errors.extend(errors.cpu().numpy())
            list_inputs.extend(inputs.cpu().numpy())

    # Convert stored results into a Pandas DataFrame for easy analysis
    # Decode tokenized text back into human-readable text
    df_results = pd.DataFrame({
        "True_Label": list_labels,
        "Predicted": list_predictions,
        "Error": list_errors,
        "Inputs": list_inputs,
        "Text" : [tokenizer.decode(input) for input in list_inputs]
    })

    # Sort the DataFrame by highest error to identify the worst predictions
    df_results_sorted = df_results.sort_values(by="Error", ascending=False)

    # Return the sorted DataFrame containing worst predictions
    return df_results_sorted

# Evaluate worst predictions on validation set
worst_predictions_val_v2 = evaluate_worst_predictions(model, val_loader, tokenizer)

# Evaluate worst predictions on training set
worst_predictions_train_v2 = evaluate_worst_predictions(model, train_loader, tokenizer)

In [244]:
print(worst_predictions_train_v2["Predicted"].value_counts())
print()
print("Previous results:")
print() #Let's compare with previous results
print(worst_predictions_train["Predicted"].value_counts())

Predicted
0    3859
1     598
Name: count, dtype: int64

Previous results:

Predicted
0    3878
1     579
Name: count, dtype: int64


In [245]:
worst_predictions_train_v2.head(10)

,True_Label,Predicted,Error,Inputs,Text
13,1,0,1.0,"[17400, 279, 24519, 46433, 126, 231, 19321, 12...",Win the newest ÂÃÃHarry Potter and the Orde...
1765,1,0,1.0,"[5519, 499, 3596, 5406, 430, 994, 499, 2351, 1...","Do you ever notice that when you're driving, a..."
896,1,0,1.0,"[13347, 41346, 1202, 60470, 11, 1268, 436, 577...","Hi babe its Chloe, how r u? I was smashed on s..."
2380,1,0,1.0,"[9906, 76745, 1268, 527, 499, 3432, 30, 358, 1...",Hello darling how are you today? I would love ...
2977,0,0,0.0,"[40, 2846, 2162, 1131, 0, 0, 0, 0, 0, 0, 0, 0,...",I'm home...!!!!!!!!!!!!!!!!!!!!!!!!!!
2975,0,0,0.0,"[4438, 33304, 499, 18754, 13, 358, 40464, 3371...",How dare you stupid. I wont tell anything to y...
2974,0,0,0.0,"[1687, 2103, 389, 369, 18396, 30, 0, 0, 0, 0, ...",We still on for tonight?!!!!!!!!!!!!!!!!!!!!!!!!
2973,0,0,0.0,"[19182, 602, 690, 387, 3389, 1131, 602, 2846, ...",Hey i will be late... i'm at amk. Need to drin...
2972,0,0,0.0,"[9453, 3091, 4912, 21781, 30, 438, 636, 1063, ...",Cancel cheyyamo?and get some money back?!!!!!!...
2971,0,0,0.0,"[32, 492, 11, 1650, 757, 3131, 499, 2351, 3345...","Aight, call me once you're close!!!!!!!!!!!!!!..."


Only 4 mistakes in the predictions

In [246]:
print(classification_report(worst_predictions_train_v2["True_Label"], worst_predictions_train_v2["Predicted"]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3855
           1       1.00      0.99      1.00       602

    accuracy                           1.00      4457
   macro avg       1.00      1.00      1.00      4457
weighted avg       1.00      1.00      1.00      4457



In [248]:
mat = confusion_matrix(worst_predictions_train_v2["True_Label"], worst_predictions_train_v2["Predicted"])

labels = df_spam["label_string"].unique()
df_mat = pd.DataFrame(mat, index=labels, columns=labels)

fig = px.imshow(df_mat, text_auto=True, color_continuous_scale=px.colors.sequential.Agsunset,
    labels=dict(x="Prediction", y="Real", color="Nombre"))
fig.update_coloraxes(showscale=False)
fig.show()

Only no real ham message was classified as "spam" in the train predictions. Which is a great result.

In [249]:
worst_predictions_val_v2.head(10) #Spam = 1 ; Ham = 0

,True_Label,Predicted,Error,Inputs,Text
542,1,0,1.0,"[29089, 499, 1093, 311, 1518, 856, 20572, 2206...",Would you like to see my XXX pics they are so ...
168,1,0,1.0,"[25821, 602, 617, 2834, 3243, 287, 1396, 220, ...",Money i have won wining number 946 wot do i do...
746,0,0,0.0,"[24220, 1070, 596, 5115, 264, 2766, 2163, 11, ...","Yeah there's quite a bit left, I'll swing by t..."
745,0,0,0.0,"[31082, 602, 8101, 13, 358, 1205, 64, 656, 296...",Later i guess. I needa do mcat study too.!!!!!...
744,1,1,0.0,"[1539, 38, 1863, 0, 8155, 9178, 596, 4128, 503...",URGENT! Last weekend's draw shows that you hav...
743,0,0,0.0,"[12174, 5509, 497, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Oh ok..!!!!!!!!!!!!!!!!!!!!!!!!!!!
742,1,1,0.0,"[44891, 432, 1753, 51, 5338, 1495, 35913, 311,...",FREE RINGTONE text FIRST to 87131 for a poly o...
741,0,0,0.0,"[19701, 11, 358, 3358, 1650, 3010, 0, 0, 0, 0,...","Sorry, I'll call later!!!!!!!!!!!!!!!!!!!!!!!!"
740,0,0,0.0,"[46864, 420, 14297, 82508, 59478, 38868, 40835...","Say this slowly.? GOD,I LOVE YOU &amp; I NEED ..."
747,0,0,0.0,"[2822, 433, 574, 26765, 22371, 8945, 0, 8489, ...",No it was cancelled yeah baby! Well that sound...


2 mispredicitons in the validation set.

In [250]:
mat = confusion_matrix(worst_predictions_val_v2["True_Label"], worst_predictions_val_v2["Predicted"])

labels = df_spam["label_string"].unique()
df_mat = pd.DataFrame(mat, index=labels, columns=labels)

fig = px.imshow(df_mat, text_auto=True, color_continuous_scale=px.colors.sequential.Agsunset,
    labels=dict(x="Prediction", y="Real", color="Nombre"))
fig.update_coloraxes(showscale=False)
fig.show()

No false positive here, which is great news. A few spam message still manage to go through the net, but in this instance, it seems more important to priorize false positives in the outcome than false negatives, as we wouldn't want clients sending real messages not to them them received as they'd be wrongly detected as spam messages.